# Day 1: Introduction to Generative AI

---

## 🧠 What is Generative AI?

Generative AI (GenAI) is designed to **think like a human** and generate answers to questions by:
* **Analyzing patterns** in training data
* **Using live web search** for current information
* **Predicting word-by-word** to construct responses

### The Core Mechanism: Word-by-Word Prediction
* GenAI doesn't generate the entire answer at once
* It predicts **one word at a time**, using previous words as context
* The process repeats: predict next word → add to context → predict again
* Continues until a complete response is generated

**Analogy:** Like finishing someone's sentence, but doing it over and over until the full thought is complete.

---

## 🔢 Tokenization: How Computers Understand Words

### The Problem
Computers only understand **numbers**, not words. So how does GenAI process language?

### The Solution: Tokenization

**Tokenization** converts text into numbers that computers can process:

| Step | Process | Example |
| --- | --- | --- |
| 1 | Original text | `"Hi, how are you"` |
| 2 | Split into tokens | `["Hi", ",", "how", "are", "you"]` |
| 3 | Convert to token IDs | `[26, 11, 352, 3242, 643]` |
| 4 | GenAI predicts next ID | `[26, 11, 352, 3242, 643, 23523]` |
| 5 | Convert back to text | `"Hi, how are you doing"` |

### Key Points About Tokens
* **Token IDs are tokenizer-specific** — not universal across all models
* Different models (GPT, Llama, etc.) may assign different IDs to the same word
* Tokens can be whole words, parts of words, or punctuation

---

## 🤖 LLMs: Large Language Models

### What Are LLMs?
* **LLM** = Large Language Model
* **GPT** = Generative Pre-trained Transformer (a type of LLM)
* LLMs use Generative AI to predict the next words in a sequence

### How LLMs Learn
* Trained **periodically** on massive datasets
* Learn patterns from text: grammar, facts, reasoning, style
* Training data has a cutoff date — they don't know events after training

### Key Limitation
**LLMs can only predict based on their training data patterns**
* If a pattern exists in training → LLM can predict
* If a pattern doesn't exist → LLM struggles or needs external help

---

## 🛠️ External Tools: When LLMs Need Help

### Why External Tools?
LLMs have knowledge limitations. When they can't answer from training data, they use **external tools**.

### Example 1: Simple Math (No Tool Needed)
* **Question:** `2 + 2`
* **What happens:** Pattern exists in training data
* **Result:** LLM directly predicts `4`

### Example 2: Complex Math (Tool Needed)
* **Question:** `12345234523452 × 2134123513251`
* **What happens:**
  1. LLM recognizes this pattern likely doesn't exist in training
  2. LLM writes Python code: `12345234523452 * 2134123513251`
  3. External Python executor runs the code
  4. LLM receives the result and returns it to the user
* **Result:** Accurate calculation using external computation

### Types of External Tools
* **Code executors** (Python, JavaScript, etc.)
* **Search engines** (for current information)
* **Databases** (for retrieving specific data)
* **APIs** (for specialized tasks)

---

## 🍓 Real Example: The "Strawberry" Problem

### The Challenge
**Question:** How many times does the letter 'r' appear in the word "strawberry"?

### What Went Wrong
* **LLM response:** "2"
* **Actual answer:** 3 (st**r**awbe**rr**y)
* **Why?** The model was trained incorrectly for this specific word

### The Solution
1. **User prompts:** "Calculate it properly using external help"
2. **LLM recognizes:** Need to verify programmatically
3. **LLM writes code:** 
   ```python
   word = "strawberry"
   count = word.count('r')
   print(count)  # Output: 3
   ```
4. **External tool executes** the code
5. **Correct answer:** 3

### Key Lesson
LLMs can make mistakes with training data. External tools provide **verification and accuracy** when needed.

---

## 💻 Today's Code Implementation

### What We Built
A **ChatSession class** that implements a stateful conversation with an LLM using the **Groq API**.

### Key Features

| Feature | Description |
| --- | --- |
| **Conversation history** | Stores all messages (system, user, assistant) |
| **System prompt** | Sets the assistant's behavior and personality |
| **Interactive loop** | Continuous chat until user exits |
| **Commands** | `history`, `clear`, `quit`/`exit` |

### How It Works
1. **Initialize** with API key, model, and system prompt
2. **Send message** → adds to history → sends to LLM
3. **Get response** → adds to history → displays to user
4. **History persists** → LLM remembers previous conversation

### Why History Matters
Without history, the LLM treats each message as a **new conversation** — it won't remember what you said before. Our implementation maintains **context** across turns.

---

## 📝 Key Takeaways

1. **GenAI predicts word-by-word** using patterns from training data
2. **Tokenization** converts words ↔ numbers for computer processing
3. **LLMs** are powerful but limited to their training data
4. **External tools** extend LLM capabilities beyond training knowledge
5. **Conversation history** enables contextual, multi-turn interactions

In [0]:
# Install package
%pip install groq

from groq import Groq
import sys

# Add user folder to Python path to import config
sys.path.insert(0, '/Users/vinith.singareddy@gmail.com')

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# Initialize client
client = Groq(api_key=GROQ_API_KEY)

# Call model
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},     {"role": "user", "content": "my name vinith"}
    ],
    temperature=0.7,  # If set, partial message deltas will be sent.
    stream=True,
)

# Print the incremental deltas returned by the LLM.
for chunk in response:
    print(chunk.choices[0].delta.content, end="")

In [0]:
%pip install groq 

import sys
from groq import Groq

# Add user folder to Python path to import config
sys.path.insert(0, '/Users/vinith.singareddy@gmail.com')

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# ═══════════════════════════════════════════════════════════════
# ChatSession — stores full conversation history so the LLM
# remembers everything said in previous turns
# ═══════════════════════════════════════════════════════════════

class ChatSession:
    def __init__(self, api_key: str, model: str = "llama-3.3-70b-versatile", system_prompt: str = "You are a helpful assistant."):
        self.client = Groq(api_key=api_key)
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message: str) -> str:
        """Send a message and get a response. History is stored automatically."""
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.5,
            max_completion_tokens=1024,
        )
        assistant_reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_reply})
        return assistant_reply

    def show_history(self):
        """Print the full conversation history."""
        for msg in self.messages:
            role = msg["role"].upper()
            print(f"[{role}]: {msg['content']}")
            print()

    def clear(self):
        """Reset history (keep only system prompt)."""
        self.messages = [self.messages[0]]
        print("✅ History cleared. Starting fresh.\n")


# ═══════════════════════════════════════════════════════════════
# Interactive Chat Loop
# ═══════════════════════════════════════════════════════════════

# Initialize ChatSession using API key from config file
session = ChatSession(
    api_key=GROQ_API_KEY,
    system_prompt="You are a helpful assistant. Remember everything the user tells you."
)

print("=" * 60)
print("🤖 Interactive Chat (with full history)")
print("=" * 60)
print("Commands:")
print("  Type 'quit' or 'exit'    → stop the chat")
print("  Type 'history'           → show full conversation history")
print("  Type 'clear'             → reset memory and start fresh")
print("=" * 60 + "\n")

while True:
    user_input = input("🗣️  Ask me a question: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit"):
        print("\n" + "=" * 60)
        print(f"👋 Chat ended. Total turns: {(len(session.messages) - 1) // 2}")
        print("=" * 60)
        print("\n📜 Final conversation history:")
        session.show_history()
        break

    if user_input.lower() == "history":
        print()
        session.show_history()
        continue

    if user_input.lower() == "clear":
        session.clear()
        continue

    reply = session.chat(user_input)
    print(f"🤖 AI: {reply}\n")